# FoodFleet: Python DB-API and analysis

Connect to MySQL with environment-based credentials, query the FoodFleet schema, and plot the four charts from the project presentation. The checked-in SQL fixture is synthetic and does not reproduce the presentation's reported aggregate figures.

In [ ]:
import os
import mysql.connector
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
conn = mysql.connector.connect(
    host=os.getenv("FOODFLEET_DB_HOST", "localhost"),
    port=int(os.getenv("FOODFLEET_DB_PORT", "3306")),
    user=os.getenv("FOODFLEET_DB_USER", "root"),
    password=os.getenv("FOODFLEET_DB_PASSWORD", ""),
    database=os.getenv("FOODFLEET_DB_NAME", "foodfleet"),
)

In [ ]:
queries = {
    "orders_by_hour": """
        SELECT HOUR(placed_at) AS hr, COUNT(*) AS orders
        FROM orders GROUP BY hr ORDER BY hr
    """,
    "restaurant_revenue": """
        SELECT restaurant_name, revenue
        FROM v_restaurant_revenue
        ORDER BY revenue DESC LIMIT 10
    """,
    "orders_by_cuisine": """
        SELECT c.cuisine_name, COUNT(*) AS orders
        FROM orders AS o
        JOIN restaurants AS r ON r.restaurant_id = o.restaurant_id
        JOIN cuisines AS c ON c.cuisine_id = r.cuisine_id
        GROUP BY c.cuisine_id, c.cuisine_name
        ORDER BY orders DESC LIMIT 10
    """,
    "delivery_time_by_hour": """
        SELECT HOUR(placed_at) AS hr,
               AVG(TIMESTAMPDIFF(MINUTE, placed_at, delivered_at)) AS avg_delivery_min
        FROM orders
        WHERE status = 'DELIVERED' AND delivered_at IS NOT NULL
        GROUP BY hr ORDER BY hr
    """,
}

try:
    results = {name: pd.read_sql(query, conn) for name, query in queries.items()}
finally:
    conn.close()

results["orders_by_hour"]

In [ ]:
charts = [
    ("orders_by_hour", "hr", "orders", "Orders by hour of day"),
    ("restaurant_revenue", "restaurant_name", "revenue", "Top restaurants by revenue"),
    ("orders_by_cuisine", "cuisine_name", "orders", "Orders by cuisine"),
    ("delivery_time_by_hour", "hr", "avg_delivery_min", "Average delivery time by hour"),
]

for key, x_column, y_column, title in charts:
    data = results[key]
    if data.empty:
        print(f"{title}: query returned no rows")
        continue
    ax = data.plot(x=x_column, y=y_column, kind="bar", legend=False, title=title, figsize=(9, 5))
    ax.set_xlabel(x_column.replace("_", " ").title())
    ax.set_ylabel(y_column.replace("_", " ").title())
    plt.tight_layout()
    plt.show()

In [ ]:
for name, frame in results.items():
    print(f"\n{name.replace('_', ' ').title()}")
    display(frame)